In [1]:
import os
from pathlib import Path
REPO = Path.cwd().resolve().parents[1]          # docs/notebooks -> repo root
GOLDEN = Path(os.environ.get("CAL_DISP_GOLDEN_DIR", REPO / "test_golden"))
SCRATCH = Path(os.environ.get("TMPDIR", REPO / "tmp")) / "cal_disp_notebooks"; SCRATCH.mkdir(parents=True, exist_ok=True)

# Fix 2 — one pytest configuration

**Defect.** The repository had two pytest configurations that disagreed:
`tests/pytest.ini` and `[tool.pytest.ini_options]` in `pyproject.toml` (line 126).
`pytest tests` — what `.github/workflows/test.yml:26` ran — starts its config search in
`tests/`, finds `tests/pytest.ini` first and never reads `pyproject.toml`: CI silently lost
`filterwarnings = error`, the fixed random seed (`--randomly-seed=1234`) and the
`--ignore` list. Plain `pytest` from the root used `pyproject.toml` instead, so local and CI
runs behaved differently. The `[coverage:run]` and `[mypy]` sections of `tests/pytest.ini`
were read by no tool.

**Fix.** `tests/pytest.ini` is deleted; its useful parts (`--strict-markers`,
`--strict-config`, `-ra`, the marker declarations, the `shapely.geos` ignore) are merged into
`pyproject.toml`; `testpaths = ["tests"]` is set; CI runs `pytest` from the repository root;
the full-workflow subprocess test in `tests/test_run_environment.py` carries the (previously
declared but unused) `slow` marker and still runs by default.

**Demonstrated here.** (1) The old layout, rebuilt in scratch from `gamma-release`: the two
invocations pick different config files and a test that emits a `DeprecationWarning` passes
under `pytest tests` but fails under `pytest`. (2) The fixed repository: both invocations
report `configfile: pyproject.toml` and collect the same tests; `filterwarnings = error` and
the markers are in effect either way.

In [2]:
import shutil
import subprocess
import sys


def pytest_run(cwd, *args):
    """Run pytest in a subprocess; return (header lines, last line)."""
    proc = subprocess.run(
        [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", "--color=no", *args],
        cwd=cwd, capture_output=True, text=True,
    )
    lines = proc.stdout.splitlines()
    header = [ln for ln in lines if ln.startswith(("rootdir:", "configfile:", "testpaths:"))]
    summary = next((ln.strip("= ") for ln in reversed(lines) if ln.strip()), "")
    return header, summary, proc


def show(label, header, summary):
    print(f"$ {label}")
    for ln in header:
        print("   ", ln.replace(str(SCRATCH), "<scratch>").replace(str(REPO), "<repo>"))
    print("    ->", summary, "\n")

## Before: the `gamma-release` layout

A miniature project with the two original config files (`git show d0901b3:...`, `gamma-release` before this fix) and one
test that emits a `DeprecationWarning`.

In [3]:
def git_show(path):
    return subprocess.run(
        ["git", "show", f"d0901b3:{path}"], cwd=REPO, check=True, capture_output=True, text=True
    ).stdout


WARNING_TEST = """import warnings


def test_emits_deprecation_warning():
    warnings.warn("this API is deprecated", DeprecationWarning)
"""

old = SCRATCH / "09_pytest_config" / "old"
shutil.rmtree(old.parent, ignore_errors=True)
(old / "tests").mkdir(parents=True)
(old / "pyproject.toml").write_text(git_show("pyproject.toml"))
(old / "tests" / "pytest.ini").write_text(git_show("tests/pytest.ini"))
(old / "tests" / "test_warning.py").write_text(WARNING_TEST)

for args in (["tests"], []):
    header, summary, _ = pytest_run(old, *args)
    show("pytest " + " ".join(args), header, summary)

$ pytest tests
    rootdir: <scratch>/09_pytest_config/old/tests
    configfile: pytest.ini
    -> 1 passed, 1 warning in 0.04s 



$ pytest 
    rootdir: <scratch>/09_pytest_config/old
    configfile: pyproject.toml
    -> 1 failed in 0.07s 



`pytest tests` (the CI command) used `tests/pytest.ini` with `rootdir: .../tests`: the
warning is only counted and the test passes. `pytest` used `pyproject.toml`, where
`filterwarnings = error` turns the same warning into a failure.

## After: the same miniature project with the fixed configuration

In [4]:
new = SCRATCH / "09_pytest_config" / "new"
(new / "tests").mkdir(parents=True)
shutil.copy(REPO / "pyproject.toml", new / "pyproject.toml")
(new / "tests" / "test_warning.py").write_text(WARNING_TEST)
print("tests/pytest.ini in the repository:", (REPO / "tests" / "pytest.ini").exists(), "\n")

results = []
for args in (["tests"], []):
    header, summary, _ = pytest_run(new, *args)
    show("pytest " + " ".join(args), header, summary)
    results.append((header, summary))
assert all(any("configfile: pyproject.toml" in ln for ln in h) for h, _ in results)
assert all("1 failed" in s for _, s in results)  # filterwarnings=error in both invocations

tests/pytest.ini in the repository: False 



$ pytest tests
    rootdir: <scratch>/09_pytest_config/new
    configfile: pyproject.toml
    -> 1 failed in 0.07s 



$ pytest 
    rootdir: <scratch>/09_pytest_config/new
    configfile: pyproject.toml
    testpaths: tests
    -> 1 failed in 0.07s 



## The repository itself

Collection only (`--collect-only`), both ways.

In [5]:
import re

collected = []
for args in (["tests"], []):
    header, summary, _ = pytest_run(REPO, "--collect-only", *args)
    show("pytest --collect-only " + " ".join(args), header, summary)
    collected.append(([ln for ln in header if not ln.startswith("testpaths")], int(re.search(r"(\d+) tests? collected", summary).group(1))))
assert collected[0] == collected[1]
assert any("configfile: pyproject.toml" in ln for ln in collected[0][0])
print("same rootdir, same config file, same", collected[0][1], "tests")

$ pytest --collect-only tests
    rootdir: <repo>
    configfile: pyproject.toml
    -> 478 tests collected in 1.84s 



$ pytest --collect-only 
    rootdir: <repo>
    configfile: pyproject.toml
    testpaths: tests
    -> 478 tests collected in 2.01s 

same rootdir, same config file, same 478 tests


In [6]:
# The effective configuration, as pytest resolved it
import tomllib

cfg = tomllib.loads((REPO / "pyproject.toml").read_text())["tool"]["pytest"]["ini_options"]
print("addopts       :", cfg["addopts"])
print("testpaths     :", cfg["testpaths"])
print("filterwarnings:")
for item in cfg["filterwarnings"]:
    print("   ", item)

_, _, proc = pytest_run(REPO, "--markers")
print("\nregistered project markers (pytest --markers):")
for ln in proc.stdout.splitlines():
    if ln.startswith(("@pytest.mark.slow", "@pytest.mark.integration", "@pytest.mark.requires_")):
        print("   ", ln)

_, summary, _ = pytest_run(REPO, "--collect-only", "-q", "-m", "slow")
print("\npytest --collect-only -q -m slow ->", summary)
_, summary, _ = pytest_run(REPO, "--collect-only", "-q", "-m", "not slow")
print("pytest --collect-only -q -m 'not slow' ->", summary)

addopts       : -ra --strict-markers --strict-config --randomly-seed=1234 --ignore=scripts --ignore=docs --ignore=data
testpaths     : ['tests']
filterwarnings:
    error
    ignore:The 'shapely.geos' module is deprecated:DeprecationWarning
    ignore:.*pkg_resources.*
    ignore:h5py is running against HDF5.*:UserWarning
    ignore:numpy.ndarray size changed, may indicate binary incompatibility:RuntimeWarning
    ignore:Use `@` matmul instead of `\*` mul operator:PendingDeprecationWarning
    ignore:Exception ignored in.* <_io.FileIO.*:pytest.PytestUnraisableExceptionWarning



registered project markers (pytest --markers):
    @pytest.mark.slow: full-workflow subprocess tests (deselect with -m 'not slow')
    @pytest.mark.integration: integration tests requiring external data or services
    @pytest.mark.requires_earthdata: tests requiring Earthdata credentials
    @pytest.mark.requires_network: tests requiring network access



pytest --collect-only -q -m slow -> 1/478 tests collected (477 deselected) in 2.02s


pytest --collect-only -q -m 'not slow' -> 477/478 tests collected (1 deselected) in 1.90s
